# 09 — Projection Threshold Boundary v2

**Empirical phase boundary for distributed residue consistency**

Notebook 09 v2 identifies the projection threshold boundary for distributed residue constraint systems.

Core question:

```text
For each link-noise level, how much projection success is required to keep the system phase-locked?
```

This notebook supports the paper sections:

- Projection Thresholds
- Scaling Law
- Phase Structure

## Outputs

```text
figures/projection_threshold_phase_diagram.png
figures/projection_threshold_curve.png
figures/projection_threshold_theory_overlay.png
figures/projection_threshold_scaling.png

results/projection_threshold_sweep.csv
results/projection_threshold_curve.csv
results/projection_threshold_theory_overlay.csv
results/projection_threshold_scaling.csv
results/projection_threshold_summary.json

docs/notebook_09_projection_threshold_boundary.md
```

In [ ]:
from pathlib import Path
import json
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import networkx as nx
except ImportError:
    !pip -q install networkx
    import networkx as nx

SEED = 9423
random.seed(SEED)
np.random.seed(SEED)

FIG_DIR = Path("figures")
RESULTS_DIR = Path("results")
DOCS_DIR = Path("docs")

for d in [FIG_DIR, RESULTS_DIR, DOCS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

PHASE_LOCK_THRESHOLD = 24 / 25

print("Ready.")
print(f"seed = {SEED}")
print(f"phase-lock threshold = {PHASE_LOCK_THRESHOLD:.3f}")

## 1. Residue system utilities

We reuse the mod30 residue manifold:

```text
R = {1, 7, 11, 13, 17, 19, 23, 29}
```

In [ ]:
MODULUS = 30
ADMISSIBLE_RESIDUES = np.array([1, 7, 11, 13, 17, 19, 23, 29], dtype=int)
ADMISSIBLE_SET = set(ADMISSIBLE_RESIDUES.tolist())

def is_admissible_residue(r):
    return int(r % MODULUS) in ADMISSIBLE_SET

def sample_local_residues(n_samples=128, noise=0.0):
    inadmissible = np.array([r for r in range(MODULUS) if r not in ADMISSIBLE_SET], dtype=int)
    values = []
    for _ in range(n_samples):
        if np.random.rand() < noise:
            values.append(int(np.random.choice(inadmissible)))
        else:
            values.append(int(np.random.choice(ADMISSIBLE_RESIDUES)))
    return np.array(values, dtype=int)

def local_coverage_score(samples):
    present = set(int(x % MODULUS) for x in samples if is_admissible_residue(x))
    return len(present.intersection(ADMISSIBLE_SET)) / len(ADMISSIBLE_RESIDUES)

def local_validity_score(samples):
    return float(np.mean([is_admissible_residue(x) for x in samples]))

def allowed_difference_set(residues=ADMISSIBLE_RESIDUES, modulus=MODULUS):
    diffs = set()
    for a in residues:
        for b in residues:
            diffs.add(int((a - b) % modulus))
    return diffs

ALLOWED_DIFFS = allowed_difference_set()

print("allowed differences:", sorted(ALLOWED_DIFFS))
print("count:", len(ALLOWED_DIFFS))

## 2. Distributed graph and noisy links

A graph node stores local residue samples.  
A graph edge stores a noisy relation between nodes.

In [ ]:
def make_module_graph(n_modules=20, k_neighbors=4, rewiring=0.15, seed=SEED):
    if n_modules < 4:
        raise ValueError("n_modules must be >= 4")
    k = min(k_neighbors, n_modules - 1)
    if k % 2 == 1:
        k += 1
    return nx.watts_strogatz_graph(n=n_modules, k=k, p=rewiring, seed=seed)

def assign_node_residue_samples(G, n_samples=128, local_noise=0.0):
    for node in G.nodes:
        samples = sample_local_residues(n_samples=n_samples, noise=local_noise)
        G.nodes[node]["samples"] = samples
        G.nodes[node]["representative"] = int(np.random.choice(samples))
        G.nodes[node]["coverage"] = local_coverage_score(samples)
        G.nodes[node]["validity"] = local_validity_score(samples)
    return G

def evaluate_links(G, link_noise=0.0, allowed_diffs=ALLOWED_DIFFS, seed=None):
    rng = np.random.default_rng(seed)
    for u, v in G.edges:
        ru = int(G.nodes[u]["representative"] % MODULUS)
        rv = int(G.nodes[v]["representative"] % MODULUS)
        corrupted = bool(rng.random() < link_noise)
        observed_rv = rv
        if corrupted:
            observed_rv = int(rng.integers(0, MODULUS))
        diff = int((ru - observed_rv) % MODULUS)
        consistent = diff in allowed_diffs
        G.edges[u, v]["corrupted"] = corrupted
        G.edges[u, v]["observed_diff"] = diff
        G.edges[u, v]["consistent"] = bool(consistent)
    return G

def link_consistency_score(G):
    if G.number_of_edges() == 0:
        return 1.0
    return float(np.mean([G.edges[e]["consistent"] for e in G.edges]))

def global_stability_score(G):
    H = nx.Graph()
    H.add_nodes_from(G.nodes)
    H.add_edges_from([e for e in G.edges if G.edges[e]["consistent"]])
    if H.number_of_nodes() == 0:
        return 0.0
    largest = max((len(c) for c in nx.connected_components(H)), default=0)
    return float(largest / H.number_of_nodes())

def cgcs_score(G):
    local_coverage = float(np.mean([G.nodes[n]["coverage"] for n in G.nodes]))
    local_validity = float(np.mean([G.nodes[n]["validity"] for n in G.nodes]))
    link_consistency = link_consistency_score(G)
    global_stability = global_stability_score(G)
    cgcs = local_coverage * local_validity * link_consistency * global_stability
    return {
        "local_coverage": local_coverage,
        "local_validity": local_validity,
        "link_consistency": link_consistency,
        "global_stability": global_stability,
        "cgcs": float(cgcs),
    }

## 3. Imperfect projection

Projection maps inconsistent edges back toward valid residue-difference constraints.

```text
with probability p_success → fix edge
otherwise → edge remains inconsistent
```

In [ ]:
def nearest_allowed_difference(diff, allowed_diffs=ALLOWED_DIFFS, modulus=MODULUS):
    allowed = np.array(sorted(list(allowed_diffs)), dtype=int)
    def cyclic_distance(a, b):
        raw = abs(int(a) - int(b)) % modulus
        return min(raw, modulus - raw)
    distances = np.array([cyclic_distance(diff, a) for a in allowed])
    return int(allowed[np.argmin(distances)])

def project_inconsistent_links_imperfect(G, allowed_diffs=ALLOWED_DIFFS, p_success=0.9, seed=None):
    rng = np.random.default_rng(seed)
    H = G.copy()
    n_attempted = 0
    n_succeeded = 0
    total_edges = H.number_of_edges()

    for u, v in H.edges:
        observed = int(H.edges[u, v].get("observed_diff", 0))
        was_consistent = bool(H.edges[u, v].get("consistent", True))
        if was_consistent:
            H.edges[u, v]["projection_attempted"] = False
            H.edges[u, v]["projection_succeeded"] = False
            H.edges[u, v]["projected_diff"] = observed
            H.edges[u, v]["consistent_after_projection"] = True
        else:
            n_attempted += 1
            succeeded = bool(rng.random() < p_success)
            H.edges[u, v]["projection_attempted"] = True
            H.edges[u, v]["projection_succeeded"] = succeeded
            if succeeded:
                projected = nearest_allowed_difference(observed, allowed_diffs=allowed_diffs)
                H.edges[u, v]["projected_diff"] = projected
                H.edges[u, v]["consistent_after_projection"] = True
                n_succeeded += 1
            else:
                H.edges[u, v]["projected_diff"] = observed
                H.edges[u, v]["consistent_after_projection"] = False
        H.edges[u, v]["consistent"] = bool(H.edges[u, v]["consistent_after_projection"])

    H.graph["projection_attempt_rate"] = n_attempted / total_edges if total_edges else 0.0
    H.graph["projection_success_rate"] = n_succeeded / n_attempted if n_attempted else 1.0
    H.graph["n_projection_attempted"] = n_attempted
    H.graph["n_projection_succeeded"] = n_succeeded
    return H

def effective_cgcs(after_cgcs, projection_rate, alpha=0.5):
    penalty = max(0.0, 1.0 - alpha * projection_rate)
    return float(after_cgcs * penalty)

## 4. Threshold sweep

Sweep:

- link noise
- projection success probability

Measure:

- CGCS after projection
- effective CGCS
- projection rate
- phase class

In [ ]:
def classify_phase(effective_score, stable_threshold=PHASE_LOCK_THRESHOLD, degraded_threshold=0.50):
    if effective_score >= stable_threshold:
        return "stable"
    if effective_score >= degraded_threshold:
        return "cost_limited"
    return "degraded"

def run_threshold_experiment(
    n_modules=20,
    k_neighbors=4,
    rewiring=0.15,
    n_samples=128,
    local_noise=0.0,
    link_noise=0.0,
    p_success=0.9,
    alpha=0.5,
    seed=SEED,
):
    G = make_module_graph(n_modules=n_modules, k_neighbors=k_neighbors, rewiring=rewiring, seed=seed)
    G = assign_node_residue_samples(G, n_samples=n_samples, local_noise=local_noise)
    G = evaluate_links(G, link_noise=link_noise, seed=seed + 99)
    before = cgcs_score(G)
    projected = project_inconsistent_links_imperfect(G, p_success=p_success, seed=seed + 199)
    after = cgcs_score(projected)
    projection_rate = projected.graph.get("projection_attempt_rate", 0.0)
    projection_success_rate = projected.graph.get("projection_success_rate", 1.0)
    eff = effective_cgcs(after["cgcs"], projection_rate, alpha=alpha)
    phase = classify_phase(eff)
    return {
        "n_modules": int(n_modules),
        "local_noise": float(local_noise),
        "link_noise": float(link_noise),
        "p_success": float(p_success),
        "alpha": float(alpha),
        "before_cgcs": before["cgcs"],
        "before_link_consistency": before["link_consistency"],
        "before_global_stability": before["global_stability"],
        "after_cgcs": after["cgcs"],
        "after_link_consistency": after["link_consistency"],
        "after_global_stability": after["global_stability"],
        "projection_rate": float(projection_rate),
        "projection_success_rate": float(projection_success_rate),
        "effective_cgcs": float(eff),
        "phase": phase,
    }

def run_threshold_sweep(
    n_modules=20,
    link_noise_values=np.linspace(0.0, 0.75, 26),
    p_success_values=np.linspace(0.0, 1.0, 26),
    repeats=12,
    alpha=0.5,
):
    rows = []
    for link_noise in link_noise_values:
        for p_success in p_success_values:
            for rep in range(repeats):
                seed = SEED + rep * 10000 + int(link_noise * 1000) + int(p_success * 1000) + n_modules
                rows.append(
                    run_threshold_experiment(
                        n_modules=n_modules,
                        link_noise=float(link_noise),
                        p_success=float(p_success),
                        alpha=alpha,
                        seed=seed,
                    )
                )
    return pd.DataFrame(rows)

threshold_df = run_threshold_sweep()
threshold_csv = RESULTS_DIR / "projection_threshold_sweep.csv"
threshold_df.to_csv(threshold_csv, index=False)

threshold_summary = (
    threshold_df
    .groupby(["link_noise", "p_success"], as_index=False)
    .mean(numeric_only=True)
)

print(threshold_df.head())
print(f"saved: {threshold_csv}")
print("rows:", len(threshold_df))

## 5. Projection threshold phase diagram

This is the main Notebook 09 figure.

In [ ]:
phase_grid = threshold_summary.pivot_table(
    index="p_success",
    columns="link_noise",
    values="effective_cgcs",
    aggfunc="mean",
)

x_values = phase_grid.columns.values
y_values = phase_grid.index.values
Z = phase_grid.values

plt.figure(figsize=(9, 5.8))
im = plt.imshow(
    Z,
    aspect="auto",
    origin="lower",
    extent=[x_values.min(), x_values.max(), y_values.min(), y_values.max()],
    vmin=0,
    vmax=1,
)
plt.colorbar(im, label="effective CGCS")

CS = plt.contour(
    x_values,
    y_values,
    Z,
    levels=[PHASE_LOCK_THRESHOLD],
    linewidths=2,
)
plt.clabel(CS, inline=True, fontsize=9, fmt={PHASE_LOCK_THRESHOLD: "24/25 threshold"})

plt.xlabel("link noise")
plt.ylabel("projection success probability")
plt.title("Projection threshold phase diagram")
plt.tight_layout()

phase_fig = FIG_DIR / "projection_threshold_phase_diagram.png"
plt.savefig(phase_fig, dpi=180, bbox_inches="tight")
plt.show()

print(f"saved: {phase_fig}")

## 6. Extract threshold curve

For each link-noise value, find the smallest projection success probability that reaches the phase-lock threshold.

In [ ]:
def extract_threshold_curve(summary_df, threshold=PHASE_LOCK_THRESHOLD):
    rows = []
    for link_noise, group in summary_df.groupby("link_noise"):
        group = group.sort_values("p_success")
        stable = group[group["effective_cgcs"] >= threshold]
        if len(stable) == 0:
            p_required = np.nan
            achieved_score = np.nan
        else:
            row = stable.iloc[0]
            p_required = float(row["p_success"])
            achieved_score = float(row["effective_cgcs"])
        rows.append(
            {
                "link_noise": float(link_noise),
                "p_required": p_required,
                "achieved_score": achieved_score,
                "threshold": float(threshold),
            }
        )
    return pd.DataFrame(rows)

threshold_curve = extract_threshold_curve(threshold_summary)
threshold_curve_csv = RESULTS_DIR / "projection_threshold_curve.csv"
threshold_curve.to_csv(threshold_curve_csv, index=False)

print(threshold_curve.head())
print(f"saved: {threshold_curve_csv}")

In [ ]:
plt.figure(figsize=(8.5, 5.2))

valid_curve = threshold_curve.dropna(subset=["p_required"]).copy()
plt.plot(
    valid_curve["link_noise"],
    valid_curve["p_required"],
    marker="o",
    linewidth=3,
    label="empirical threshold",
)

plt.xlabel("link noise")
plt.ylabel("required projection success")
plt.ylim(-0.02, 1.05)
plt.title("Projection success required for phase-lock")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()

curve_fig = FIG_DIR / "projection_threshold_curve.png"
plt.savefig(curve_fig, dpi=180, bbox_inches="tight")
plt.show()

print(f"saved: {curve_fig}")

## 7. Theory overlay

A simple threshold model says recovery becomes difficult as inconsistent-link pressure grows:

```text
p_required ≈ rho_crit / rho
```

In [ ]:
rho_est = (
    threshold_df
    .groupby("link_noise", as_index=False)
    .agg({"projection_rate": "mean"})
    .rename(columns={"projection_rate": "rho_est"})
)

theory_curve = threshold_curve.merge(rho_est, on="link_noise", how="left")
valid = theory_curve.dropna(subset=["p_required", "rho_est"]).copy()
valid = valid[valid["rho_est"] > 0]

if len(valid) > 0:
    rho_crit_fit = float(np.median(valid["p_required"] * valid["rho_est"]))
else:
    rho_crit_fit = 0.1

theory_curve["p_theory"] = rho_crit_fit / theory_curve["rho_est"].replace(0, np.nan)
theory_curve["p_theory"] = theory_curve["p_theory"].clip(0, 1)

theory_csv = RESULTS_DIR / "projection_threshold_theory_overlay.csv"
theory_curve.to_csv(theory_csv, index=False)

print("rho_crit_fit:", rho_crit_fit)
print(theory_curve.head())
print(f"saved: {theory_csv}")

In [ ]:
plt.figure(figsize=(8.5, 5.2))

valid_emp = theory_curve.dropna(subset=["p_required"]).copy()
valid_theory = theory_curve.dropna(subset=["p_theory"]).copy()

plt.plot(
    valid_emp["link_noise"],
    valid_emp["p_required"],
    marker="o",
    linewidth=3,
    label="empirical threshold",
)

plt.plot(
    valid_theory["link_noise"],
    valid_theory["p_theory"],
    linestyle="--",
    linewidth=2,
    label="simple theory overlay",
)

plt.xlabel("link noise")
plt.ylabel("required projection success")
plt.ylim(-0.02, 1.05)
plt.title("Empirical threshold vs simple theory")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()

overlay_fig = FIG_DIR / "projection_threshold_theory_overlay.png"
plt.savefig(overlay_fig, dpi=180, bbox_inches="tight")
plt.show()

print(f"saved: {overlay_fig}")

## 8. Scaling comparison

Compare threshold behavior across graph sizes.

This supports the scaling-law section of the paper.

In [ ]:
def run_scaling_thresholds(
    n_values=(12, 20, 32),
    link_noise_values=np.linspace(0.0, 0.75, 16),
    p_success_values=np.linspace(0.0, 1.0, 21),
    repeats=8,
    alpha=0.5,
):
    rows = []
    for n_modules in n_values:
        print("running n_modules =", n_modules)
        df_n = run_threshold_sweep(
            n_modules=n_modules,
            link_noise_values=link_noise_values,
            p_success_values=p_success_values,
            repeats=repeats,
            alpha=alpha,
        )
        summary_n = (
            df_n
            .groupby(["n_modules", "link_noise", "p_success"], as_index=False)
            .mean(numeric_only=True)
        )
        curve_n = extract_threshold_curve(summary_n)
        curve_n["n_modules"] = n_modules
        rows.append(curve_n)
    return pd.concat(rows, ignore_index=True)

scaling_curve = run_scaling_thresholds()

scaling_csv = RESULTS_DIR / "projection_threshold_scaling.csv"
scaling_curve.to_csv(scaling_csv, index=False)

print(scaling_curve.head())
print(f"saved: {scaling_csv}")

In [ ]:
plt.figure(figsize=(8.8, 5.4))

for n_modules, group in scaling_curve.groupby("n_modules"):
    group = group.dropna(subset=["p_required"])
    plt.plot(
        group["link_noise"],
        group["p_required"],
        marker="o",
        linewidth=2,
        label=f"N={n_modules}",
    )

plt.xlabel("link noise")
plt.ylabel("required projection success")
plt.ylim(-0.02, 1.05)
plt.title("Projection threshold shifts with graph size")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()

scaling_fig = FIG_DIR / "projection_threshold_scaling.png"
plt.savefig(scaling_fig, dpi=180, bbox_inches="tight")
plt.show()

print(f"saved: {scaling_fig}")

## 9. Threshold scaling fit

The threshold curve suggests an onset region: below a critical link-noise level, little or no projection is required; above it, required projection success rises sharply.

We fit an approximate power law:

```text
p_required ≈ A · (link_noise − noise_crit)^β
```

This converts the empirical threshold curve into a compact scaling summary.

In [ ]:
# Extract valid empirical threshold points.
fit_df = threshold_curve.dropna(subset=["p_required"]).copy()
fit_df = fit_df.sort_values("link_noise")

link_noise = fit_df["link_noise"].to_numpy(dtype=float)
p_required = fit_df["p_required"].to_numpy(dtype=float)

positive = np.where(p_required > 0)[0]

if len(positive) == 0:
    noise_crit = np.nan
    beta = np.nan
    A = np.nan
    fit_points = 0
    print("No positive threshold values found; fit skipped.")
else:
    noise_crit = float(link_noise[positive[0]])

    # Fit only above onset and below saturation.
    fit_mask = (
        (link_noise > noise_crit)
        & (p_required > 0)
        & (p_required < 0.98)
    )

    x = link_noise[fit_mask] - noise_crit
    y = p_required[fit_mask]
    fit_points = int(len(x))

    if fit_points < 2:
        beta = np.nan
        A = np.nan
        print("Not enough points above critical noise for power-law fit.")
    else:
        eps = 1e-12
        logx = np.log(x + eps)
        logy = np.log(y + eps)

        coeffs = np.polyfit(logx, logy, 1)
        beta = float(coeffs[0])
        A = float(np.exp(coeffs[1]))

    print(f"Estimated critical noise ≈ {noise_crit:.4f}")
    print(f"Fitted exponent beta ≈ {beta:.4f}")
    print(f"Fitted prefactor A ≈ {A:.4f}")

scaling_fit = {
    "noise_crit": None if np.isnan(noise_crit) else float(noise_crit),
    "beta": None if np.isnan(beta) else float(beta),
    "A": None if np.isnan(A) else float(A),
    "threshold": float(PHASE_LOCK_THRESHOLD),
    "fit_points": int(fit_points),
}

scaling_fit_path = RESULTS_DIR / "projection_threshold_scaling_fit.json"
scaling_fit_path.write_text(json.dumps(scaling_fit, indent=2), encoding="utf-8")

print(f"saved: {scaling_fit_path}")

In [ ]:
plt.figure(figsize=(8.5, 5.2))

plt.scatter(
    link_noise,
    p_required,
    s=70,
    label="empirical threshold",
)

if not np.isnan(noise_crit) and not np.isnan(beta) and not np.isnan(A):
    x_fit = np.linspace(
        0,
        max(link_noise.max() - noise_crit, 1e-6),
        200,
    )

    y_fit = A * (x_fit ** beta)
    y_fit = np.clip(y_fit, 0, 1)

    plt.plot(
        noise_crit + x_fit,
        y_fit,
        linestyle="--",
        linewidth=2,
        label=f"fit: beta={beta:.2f}",
    )

    plt.axvline(
        noise_crit,
        linestyle=":",
        linewidth=2,
        label=f"critical noise≈{noise_crit:.2f}",
    )

plt.xlabel("link noise")
plt.ylabel("required projection success")
plt.ylim(-0.02, 1.05)
plt.title("Projection threshold scaling fit")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()

fit_fig = FIG_DIR / "projection_threshold_scaling_fit.png"
plt.savefig(fit_fig, dpi=180, bbox_inches="tight")
plt.show()

print(f"saved: {fit_fig}")

## Threshold scaling interpretation

The fitted curve is not meant as a universal law. It is a compact summary of this finite-state experiment.

```text
below critical noise: projection requirement is minimal
above critical noise: required projection success rises sharply
near saturation: almost perfect correction is required
```

This is the empirical version of the paper claim:

```text
projection threshold increases with link noise
```

## 10. Summary exports

In [ ]:
summary_payload = {
    "notebook": "09_projection_threshold_boundary.ipynb",
    "seed": SEED,
    "phase_lock_threshold": PHASE_LOCK_THRESHOLD,
    "modulus": MODULUS,
    "admissible_residues": ADMISSIBLE_RESIDUES.tolist(),
    "allowed_differences": sorted(list(ALLOWED_DIFFS)),
    "core_claim": (
        "Projection success threshold increases with link noise; "
        "phase-locked distributed consistency requires sufficient projection success."
    ),
    "figures": [
        "figures/projection_threshold_phase_diagram.png",
        "figures/projection_threshold_curve.png",
        "figures/projection_threshold_theory_overlay.png",
        "figures/projection_threshold_scaling.png",
        "figures/projection_threshold_scaling_fit.png",
    ],
    "results": [
        "results/projection_threshold_sweep.csv",
        "results/projection_threshold_curve.csv",
        "results/projection_threshold_theory_overlay.csv",
        "results/projection_threshold_scaling.csv",
        "results/projection_threshold_scaling_fit.json",
    ],
}

summary_path = RESULTS_DIR / "projection_threshold_summary.json"
summary_path.write_text(json.dumps(summary_payload, indent=2), encoding="utf-8")

md_lines = [
    "# Notebook 09 — Projection Threshold Boundary",
    "",
    "**Core claim:** projection success threshold increases with link noise.",
    "",
    "This notebook identifies the empirical phase boundary for distributed residue consistency.",
    "",
    "## Outputs",
    "",
    "- `figures/projection_threshold_phase_diagram.png`",
    "- `figures/projection_threshold_curve.png`",
    "- `figures/projection_threshold_theory_overlay.png`",
    "- `figures/projection_threshold_scaling.png`\n- `figures/projection_threshold_scaling_fit.png`",
    "",
    "## Takeaway",
    "",
    "Distributed consistency is not restored by projection alone. It requires projection success above a noise-dependent threshold.",
    "",
]

md_path = DOCS_DIR / "notebook_09_projection_threshold_boundary.md"
md_path.write_text("\n".join(md_lines), encoding="utf-8")

print(json.dumps(summary_payload, indent=2))
print(f"saved: {summary_path}")
print(f"saved: {md_path}")

## 11. Optional zip/export block for Colab

In [ ]:
import zipfile

zip_path = Path("notebook_09_outputs.zip")

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for folder in [FIG_DIR, RESULTS_DIR, DOCS_DIR]:
        for path in folder.rglob("*"):
            if path.is_file():
                z.write(path, path.as_posix())

print(f"created: {zip_path}")

# Optional Colab download:
# from google.colab import files
# files.download(str(zip_path))

## Final interpretation

Notebook 09 turns the paper threshold claim into an empirical object:

```text
projection success threshold = measurable boundary
```

The result supports:

```text
link noise ↑ → required projection success ↑
graph size ↑ → threshold shifts
```

This strengthens the distributed constraint-consistency framework beyond qualitative projection recovery.